In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

83

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. 

This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one. When you build your own evaluation set, assign an ID to each record in your knowledge base first.

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

### Below is an updated version of the course, using groq instad

In [10]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "Questions_schema", # Changed "Questions schema" to "Questions_schema"
            "schema": Questions.model_json_schema()
        }
    }
)

In [11]:
raw_result = json.loads(response.choices[0].message.content or "{}")
result = Questions.model_validate(raw_result)
print(result.model_dump_json(indent=2))

{
  "questions": [
    "Can I enroll in the LLM Zoomcamp now that I just found out about it?",
    "Is it still possible to join the course after it started?",
    "Am I allowed to sign up for the course at this point?",
    "If I join late, can I still earn a certificate?",
    "Do I need to submit a project to get a certificate if I enroll now?"
  ]
}


In [12]:
from groq_evaluation_utils import llm_structured

In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still enroll in the LLM Zoomcamp now that I just found out about it?', 'Is it still possible to get a certificate if I join late?', 'Do I need to submit a project to earn the certificate after joining late?', 'What’s the deadline for submitting projects to receive the certificate?', 'Are there any restrictions on joining the course after it has started?']


# Note: llama-3.3-70b-versatile does not handle json format outputs, so use openai/gpt-oss-120b

In [25]:
usage

CompletionUsage(completion_tokens=392, prompt_tokens=332, total_tokens=724, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=288, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.191409879, prompt_time=0.01733278, completion_time=0.828673483, total_time=0.846006263)

In [24]:
# Assuming 'usage' is your CompletionUsage object
print(f"Total Tokens: {usage.total_tokens}")
print(f"Completion Tokens: {usage.completion_tokens}")
print(f"Prompt Tokens: {usage.prompt_tokens}")

Total Tokens: 724
Completion Tokens: 392
Prompt Tokens: 332


In [26]:
# or:
# Convert the entire object to a dictionary
usage_dict = usage.model_dump()

# Access values like a dictionary
print(usage_dict["total_tokens"])

724


In [22]:
from groq_evaluation_utils import calc_price

In [23]:
cost = calc_price(usage)

cost

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'